# M1 — Semantic Scholar references (Colab + Drive)

Chạy **sau** notebook crawl arXiv của người 1 (`M1_Colab_Concurrent_Limited.ipynb`).
Quy trình: GitHub branch → Drive → đọc ID từ checkpoint `batch_*.tar.gz` → crawl references (resume) → xuất 2 bảng `papers` / `references` + validate.

- **1 request / publication** (S2 không phân version), delay ≥ 1s, có `S2_API_KEY` thì ổn định hơn (vào **Secrets** của Colab thêm secret tên `S2_API_KEY`; không có vẫn chạy được).
- Resume: paper đã có `references_s2.json` trên Drive sẽ bị bỏ qua, không gọi API lại.
- Lưu **toàn bộ** reference (kể cả không có arXiv ID); bảng `references` xuất ra **chỉ giữ** ref có arXiv ID.
- Lỗi từng paper (chưa index, 429 sau retry, mất mạng) được log rồi bỏ qua, không dừng cả run.

**Cần push code Python mới nhất lên nhánh của nhóm trước khi chạy notebook.**

## SET UP — Clone code từ branch GitHub và cài thư viện

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tuandatedu/arXir_project.git"  # TODO: thay URL repo nhóm
BRANCH = "M1-update"                    # TODO: tên branch thật
PROJECT = Path("/content/project")
assert "OWNER" not in REPO_URL, "Bạn chưa thay URL GitHub"

if not (PROJECT / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_m1_pipeline.py"], check=True)
print("Repo:", PROJECT, "| Branch:", BRANCH)

## SET UP — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1 — Trỏ đúng thư mục Drive của range

`RANGE_ID` và đường dẫn `DATA_DIR` phải **trùng với notebook crawl của người 1** để tìm được checkpoint. Dữ liệu references được ghi xen vào cạnh đó:

```text
DATA_DIR/
  batch_00000.tar.gz + .json     # checkpoint của người 1 (nguyên trạng)
  references/<folder>/references_s2.json   # MỌI ref đã crawl (resume tại đây)
  tables/papers.json, references.json, metrics_s2_references.json, s2_crawl_records.csv
```

In [ ]:
RANGE_ID = 5             # TODO: range của người chạy (1..5)
DATA_DIR = Path(f"/content/drive/MyDrive/arxiv-project-data/m1/range_{RANGE_ID}_dev")
REFS_ROOT = DATA_DIR / "references"
TABLES_DIR = DATA_DIR / "tables"
for path in (DATA_DIR, REFS_ROOT, TABLES_DIR):
    path.mkdir(parents=True, exist_ok=True)

manifests = sorted(DATA_DIR.glob("batch_*.json"))
assert manifests, f"Chưa có checkpoint batch_*.json trong {DATA_DIR} — chạy notebook crawl người 1 trước"
print("Range:", RANGE_ID, "| Drive:", DATA_DIR, "| checkpoints:", len(manifests))

In [ ]:
import os

# Gán trực tiếp S2_API_KEY vào biến môi trường
os.environ["S2_API_KEY"] = "s2k-2GSKU1aa1guduuqNUaFQZSXLrZ5K5RwCGbPNRBN8"
print("Đã nạp S2_API_KEY thành công!")

from src.common import config

config.WORKING_DIR = DATA_DIR
config.TABLES_DIR = TABLES_DIR
print("TABLES_DIR =", config.TABLES_DIR)

## 2 — Đọc danh sách paper từ checkpoint `batch_*.tar.gz`

Mỗi archive chứa `papers.jsonl` (metadata người 1 đã crawl). Chỉ lấy paper có `status == "ok"` trong `crawl_records.jsonl` — paper lỗi thì không có metadata để tra cứu.

In [ ]:
import json
import tarfile

def _read_jsonl_from_archive(archive: Path, name: str) -> list[dict]:
    with tarfile.open(archive, "r:gz") as tf:
        entry = tf.extractfile(name)
        if entry is None:
            raise ValueError(f"{archive.name} thiếu {name}")
        with entry:
            return [json.loads(line) for line in entry if line.strip()]

def load_papers_from_checkpoints() -> list[dict]:
    papers: dict[str, dict] = {}
    for marker in sorted(DATA_DIR.glob("batch_*.json")):
        archive = marker.with_suffix(".tar.gz")
        if not archive.exists():
            print("MISSING ARCHIVE:", archive.name)
            continue
        ok_ids = {str(r["arxiv_id"]) for r in
                  _read_jsonl_from_archive(archive, "crawl_records.jsonl")
                  if r.get("status") == "ok"}
        for row in _read_jsonl_from_archive(archive, "papers.jsonl"):
            if str(row.get("arxiv_id")) in ok_ids:
                papers[str(row["arxiv_id"])] = row
    return list(papers.values())

papers_meta = load_papers_from_checkpoints()
print(f"{len(papers_meta)} paper có metadata trong checkpoint")
print("Ví dụ:", papers_meta[0]["arxiv_id"], "|", papers_meta[0]["title"][:80])

## 3 — Crawl references (resume, có đo runtime)

Mỗi paper: `fetch_s2_references` → `s2_ref_to_local_record` → ghi `DATA_DIR/references/<folder>/references_s2.json`.
Rate limit do thư viện lo (`S2_MIN_DELAY_SEC`); mỗi publication chỉ 1 request. Run bị ngắt giữa chừng thì chạy lại cell — paper đã lưu sẽ `skipped`.

In [ ]:
import csv
import logging
import time

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from src.common.ids import folder_name_for_arxiv_id, normalize_arxiv_id
from src.common.metrics import track_run
from src.common.io import write_jsonl
from src.m1_crawling.semantic_scholar import fetch_s2_references, s2_ref_to_local_record

def refs_path_for(arxiv_id: str) -> Path:
    return REFS_ROOT / folder_name_for_arxiv_id(arxiv_id) / "references_s2.json"

results = []
started_at = time.perf_counter()
with track_run("s2_references") as metrics:
    metrics["range_id"] = RANGE_ID
    metrics["n_ids"] = len(papers_meta)
    for index, row in enumerate(papers_meta, start=1):
        base = normalize_arxiv_id(row["arxiv_id"])
        target = refs_path_for(base)
        if target.exists():
            results.append({"arxiv_id": base, "status": "skipped"})
            continue
        try:
            raw = fetch_s2_references(base)
            records = [s2_ref_to_local_record(base, item) for item in raw]
            write_jsonl(target, records)
            n_arxiv = sum(1 for r in records if r["ref_arxiv_id"])
            results.append({"arxiv_id": base, "status": "ok",
                            "n_refs": len(records), "n_arxiv": n_arxiv})
        except Exception as exc:
            print(f"LỖI {base}: {exc}")
            results.append({"arxiv_id": base, "status": "error", "error": str(exc)})
        if index % 50 == 0:
            print(f"... {index}/{len(papers_meta)}")

ok = sum(r["status"] == "ok" for r in results)
skipped = sum(r["status"] == "skipped" for r in results)
failed = sum(r["status"] == "error" for r in results)
total_refs = sum(r.get("n_refs", 0) for r in results)
total_arxiv = sum(r.get("n_arxiv", 0) for r in results)
print(f"ok={ok} skipped={skipped} error={failed} refs={total_refs} (có arXiv ID: {total_arxiv})")
print(f"Wall-clock: {time.perf_counter() - started_at:.1f}s")
print("Metrics:", {k: metrics.get(k) for k in
                   ("duration_sec", "peak_rss_bytes", "avg_rss_bytes")})

In [ ]:
# Ghi log từng paper (CSV) để đối chiếu / gửi người xử lý dữ liệu.
records_csv = TABLES_DIR / "s2_crawl_records.csv"
with records_csv.open("w", newline="", encoding="utf-8-sig") as handle:
    writer = csv.DictWriter(handle, fieldnames=["arxiv_id", "status", "n_refs", "n_arxiv", "error"])
    writer.writeheader()
    for record in results:
        writer.writerow({key: record.get(key, "") for key in writer.fieldnames})
print("Đã ghi", records_csv)

## 4 — Recovery: chạy lại phần references dở dang

Chỉ chạy cell crawl ở mục 3 thêm lần nữa là đủ: paper đã lưu sẽ `skipped`, paper lỗi/chưa tới sẽ được thử lại.
Nếu 429 dồn dập thì chờ ≥ 5 phút rồi chạy lại; **không tăng tốc hoặc mở song song**.

## 5 — Xuất 2 bảng `papers` / `references` + validate

- `papers`: metadata đọc từ checkpoint (1 hàng / publication).
- `references`: chỉ ref **có arXiv ID** (matching pool của M3); mọi ref vẫn nằm trong `references/<folder>/references_s2.json`.
Kết quả validate (thiếu thông tin / sai kiểu / trùng lặp) được in ra và ghi vào metrics.

In [ ]:
from src.common.io import read_jsonl
from src.common.schemas import PaperRow, ReferenceRow
from src.m1_crawling.tables import (
    s2_records_to_reference_rows,
    validate_m1_tables,
    write_m1_tables,
)

paper_rows: list[PaperRow] = []
reference_rows: list[ReferenceRow] = []
seen_ids: set[str] = set()
for row in papers_meta:
    base = normalize_arxiv_id(row["arxiv_id"])
    if base in seen_ids:
        continue
    seen_ids.add(base)
    paper_rows.append(PaperRow(
        arxiv_id=base,
        title=row.get("title", ""),
        authors=row.get("authors") or [],
        submission_date=row.get("submission_date", ""),
        revised_dates=row.get("revised_dates") or [],
        venue=row.get("venue"),
    ))
    target = refs_path_for(base)
    if target.exists():
        reference_rows.extend(
            s2_records_to_reference_rows(base, read_jsonl(target)))

report = validate_m1_tables(paper_rows, reference_rows)
print("validate ok =", report["ok"], "| papers =", report["n_papers"],
      "| references =", report["n_references"], "| issues =", report["n_issues"])
for issue in report["issues"][:20]:
    print(issue)

write_m1_tables(paper_rows, reference_rows)
print("papers.json     ->", TABLES_DIR / "papers.json")
print("references.json ->", TABLES_DIR / "references.json")
print("metrics          ->", TABLES_DIR / "metrics_s2_references.json")

In [ ]:
# Kiểm tra nhanh output trên Drive
from pathlib import Path
import json

n_ref_files = len(list(REFS_ROOT.glob("*/references_s2.json")))
print(f"references_s2.json đã lưu: {n_ref_files} paper")

sample = next(REFS_ROOT.glob("*/references_s2.json"), None)
if sample:
    rows = read_jsonl(sample)
    with_arxiv = sum(1 for r in rows if r.get("ref_arxiv_id"))
    print(f"Ví dụ {sample.parent.name}: {len(rows)} refs, {with_arxiv} có arXiv ID")
    print("Một record:", rows[0] if rows else None)

for name in ("papers.json", "references.json", "s2_crawl_records.csv",
             "metrics_s2_references.json"):
    path = TABLES_DIR / name
    print(f"- {name}: {path.stat().st_size / 1024:.1f} KiB" if path.exists()
          else f"- {name}: MISSING")

## Sau khi chạy

- Gửi cho người xử lý dữ liệu: `tables/papers.json`, `tables/references.json`, `tables/s2_crawl_records.csv` và thư mục `references/` trên Drive.
- `references/` chứa **mọi** reference (gồm ref không có arXiv ID) — đừng xóa trước khi submit Hugging Face ở M3.
- Chạy đầy đủ cả pipeline (source + S2 + bảng) trên máy:
  `python -m src.m1_crawling.run_m1 --ids assigned_ids.txt`
- Report Jupyter Book đọc `tables/metrics_s2_references.json` + `data/tables/metrics_m1.json` để vẽ figure lúc compile (`book/m1_report.md`).